In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# Cell 1 — run the full pipeline
import subprocess
result = subprocess.run(
    ['python', '/content/drive/MyDrive/assignment/pipeline/run_pipeline.py'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)


Traceback (most recent call last):
  File "/content/drive/MyDrive/assignment/pipeline/run_pipeline.py", line 84, in <module>
    run()
  File "/content/drive/MyDrive/assignment/pipeline/run_pipeline.py", line 38, in run
    con = duckdb.connect(DB_PATH)
          ^^^^^^^^^^^^^^^^^^^^^^^
duckdb.duckdb.IOException: IO Error: Could not set lock on file "/content/drive/MyDrive/assignment/ecommerce.duckdb": Conflicting lock is held in /usr/bin/python3.12 (PID 2303). See also https://duckdb.org/docs/stable/connect/concurrency



In [8]:
# Cell 2 — verify row counts after load
import duckdb
con = duckdb.connect('/content/drive/MyDrive/assignment/ecommerce.duckdb')

con.execute("""
    SELECT event_month, event_type, COUNT(*) AS cnt
    FROM fact_events
    GROUP BY event_month, event_type
    ORDER BY event_month, event_type
""").df()

CatalogException: Catalog Error: Table with name fact_events does not exist!
Did you mean "pg_sequences"?

LINE 3:     FROM fact_events
                 ^

In [9]:
# Cell 3 — idempotency proof: re-run and confirm counts don't change
before = con.execute("SELECT COUNT(*) FROM fact_events").fetchone()[0]

# run pipeline again on same files
subprocess.run(['python', '.../run_pipeline.py'], capture_output=True)

after = con.execute("SELECT COUNT(*) FROM fact_events").fetchone()[0]
print(f"Before re-run: {before:,}")
print(f"After  re-run: {after:,}")
print("✓ Idempotent" if before == after else "✗ Duplicates created!")

CatalogException: Catalog Error: Table with name fact_events does not exist!
Did you mean "pg_sequences"?

LINE 1: SELECT COUNT(*) FROM fact_events
                             ^

In [ ]:
# Cell 4 — print quality report
import json
with open('/content/drive/MyDrive/assignment/reports/pipeline_run.json') as f:
    log = json.load(f)
print(json.dumps(log, indent=2))